In [1]:
import json
import subprocess
import sys
from collections import Counter
import pandas as pd

print('Python:', sys.version)
print('pandas:', pd.__version__)

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:16:10) [GCC 11.2.0]
pandas: 2.3.3


## 1. Load and Inspect the Task Dataset

In [2]:
with open('../data/humaneval_subset.json') as f:
    problems = json.load(f)

df = pd.DataFrame(problems)
print(f'Total problems: {len(df)}')
print(f'\nColumns: {list(df.columns)}')
df.head(3)

Total problems: 80

Columns: ['task_id', 'bug_category', 'prompt', 'canonical_solution', 'test', 'description', 'function_length', 'complexity']


,task_id,bug_category,prompt,canonical_solution,test,description,function_length,complexity
0,HE_001,off_by_one,"def has_close_elements(numbers, threshold):\n ...","def has_close_elements(numbers, threshold):\n ...","assert has_close_elements([1.0,2.0,3.9,4.0,5.0...",Loop upper bound exceeds list length by 1,6,3
1,HE_002,off_by_one,"def intersperse(numbers, delimeter):\n resu...","def intersperse(numbers, delimeter):\n resu...","assert intersperse([1,2,3],4)==[1,4,2,4,3]",Last element not appended after loop,5,2
2,HE_003,off_by_one,def mean_absolute_deviation(numbers):\n mea...,def mean_absolute_deviation(numbers):\n mea...,"assert abs(mean_absolute_deviation([1.0,2.0,3....",Division by n-1 instead of n,3,2


In [3]:
# Verify balanced split across categories
print('Category counts:')
print(df['bug_category'].value_counts())

print('\nFunction length stats:')
print(df['function_length'].describe().round(2))

print('\nComplexity stats:')
print(df['complexity'].describe().round(2))

Category counts:
bug_category
off_by_one     20
type_error     20
scope_error    20
logic_error    20
Name: count, dtype: int64

Function length stats:
count    80.00
mean      4.68
std       2.53
min       2.00
25%       3.00
50%       4.00
75%       6.00
max      16.00
Name: function_length, dtype: float64

Complexity stats:
count    80.00
mean      2.22
std       0.90
min       1.00
25%       2.00
50%       2.00
75%       3.00
max       5.00
Name: complexity, dtype: float64


### Observations

- The dataset is **perfectly balanced**: exactly 20 problems per bug category.
- **Function lengths** range from 2 to 16 lines (mean ~4.7). Most problems are short, which keeps per-episode token cost low during Phase 2 agent runs.
- **Complexity** ranges from 1 to 5 (mean 2.2). Logic errors and scope errors contain the highest-complexity problems — this will be relevant for per-category analysis in Phase 2.
- Both buggy (`prompt`) and correct (`canonical_solution`) versions are stored. The buggy version is what the agent sees; correctness is judged by running the `test` field through pytest.

## 2. Validate the Automated Test Runner

Before any agent runs, we confirm that the test harness correctly scores success (passing test) and failure (failing test) for a sample problem.

In [4]:
import textwrap, ast, traceback

def run_test(canonical_solution: str, test_code: str) -> bool:
    """
    Execute canonical_solution + test_code in a sandboxed namespace.
    Returns True if all assertions pass, False otherwise.
    """
    namespace = {}
    try:
        exec(canonical_solution, namespace)
        exec(test_code, namespace)
        return True
    except AssertionError:
        return False
    except Exception as e:
        return False

# Test on the first 5 problems using canonical solutions (should all pass)
passed = 0
for p in problems[:5]:
    result = run_test(p['canonical_solution'], p['test'])
    status = 'PASS' if result else 'FAIL'
    print(f"{p['task_id']} [{p['bug_category']}]: {status}")
    if result: passed += 1
print(f'\nCanonical solutions: {passed}/5 passed (expected: 5/5)')

HE_001 [off_by_one]: PASS
HE_002 [off_by_one]: PASS
HE_003 [off_by_one]: PASS
HE_004 [off_by_one]: PASS
HE_005 [off_by_one]: PASS

Canonical solutions: 5/5 passed (expected: 5/5)


In [5]:
# Test on the first 5 problems using BUGGY prompts (should mostly fail)
failed = 0
for p in problems[:5]:
    result = run_test(p['prompt'], p['test'])
    status = 'PASS' if result else 'FAIL'
    print(f"{p['task_id']} [{p['bug_category']}]: {status} (buggy version)")
    if not result: failed += 1
print(f'\nBuggy versions that correctly fail: {failed}/5')

HE_001 [off_by_one]: FAIL (buggy version)
HE_002 [off_by_one]: FAIL (buggy version)
HE_003 [off_by_one]: FAIL (buggy version)
HE_004 [off_by_one]: FAIL (buggy version)
HE_005 [off_by_one]: FAIL (buggy version)

Buggy versions that correctly fail: 5/5


The test runner correctly distinguishes correct from buggy solutions. This confirms that binary success scoring will work reliably in Phase 2.

## 3. Inspect the Synthetic Memory Bank

In [6]:
with open('../data/synthetic_memories.json') as f:
    memories = json.load(f)

df_mem = pd.DataFrame(memories).drop(columns=['embedding_placeholder'])
print(f'Total memories: {len(df_mem)}')
print(f'\nColumns: {list(df_mem.columns)}')
df_mem.head(3)

Total memories: 25

Columns: ['memory_id', 'reflection', 'task_category', 'success_rate', 'times_retrieved', 'times_helpful', 'confidence', 'recency_ts']


,memory_id,reflection,task_category,success_rate,times_retrieved,times_helpful,confidence,recency_ts
0,MEM_001,"When iterating with range(len(lst)), using len...",off_by_one,0.88,17,15,0.91,1748000000
1,MEM_002,"Products must be initialised to 1, not 0. Init...",off_by_one,0.93,14,13,0.95,1748200000
2,MEM_003,When generating prefixes or suffixes with rang...,off_by_one,0.80,10,8,0.85,1747800000


In [7]:
print('Memory stats by category:')
print(df_mem.groupby('task_category')[['success_rate','confidence','times_retrieved','times_helpful']].mean().round(3))

low_sr = df_mem[df_mem['success_rate'] < 0.3]
print(f'\nLow-reliability memories (SR < 0.3): {len(low_sr)}')
print(low_sr[['memory_id','task_category','success_rate','confidence']])

Memory stats by category:
               success_rate  confidence  times_retrieved  times_helpful
task_category                                                          
logic_error           0.747       0.765           13.833         11.167
off_by_one            0.656       0.687           13.286          9.143
scope_error           0.730       0.750           17.500         13.833
type_error            0.732       0.752           16.500         13.167

Low-reliability memories (SR < 0.3): 5
   memory_id task_category  success_rate  confidence
20   MEM_021    off_by_one          0.15        0.20
21   MEM_022    type_error          0.10        0.12
22   MEM_023   scope_error          0.08        0.10
23   MEM_024   logic_error          0.12        0.14
24   MEM_025    off_by_one          0.18        0.20


### Memory Bank Design Notes

- **20 high-reliability memories** (SR >= 0.7) provide the signal that the reliability-aware retrieval policy should learn to prefer.
- **5 low-reliability memories** (SR < 0.3) serve as noise. The similarity-only baseline (Baseline B) will sometimes retrieve these over better options when the embedding similarity happens to be high — this is the *false memory rate* metric from the proposal.
- Confidence scores are intentionally correlated with success rate to simulate a well-calibrated estimator; Phase 2 will measure the actual ECE of the LLM-estimated confidence.

## 4. Qdrant Memory Schema Validation

Below we define and document the exact Qdrant payload schema used in Phase 2. We validate that all 25 synthetic memories conform to it.

In [8]:
SCHEMA = {
    'reflection':      str,
    'task_category':   str,
    'success_rate':    float,
    'times_retrieved': int,
    'times_helpful':   int,
    'confidence':      float,
    'recency_ts':      int,
}

def validate_memory(mem: dict) -> list:
    errors = []
    for field, expected_type in SCHEMA.items():
        if field not in mem:
            errors.append(f'Missing field: {field}')
        elif not isinstance(mem[field], expected_type):
            errors.append(f'Field {field}: expected {expected_type.__name__}, got {type(mem[field]).__name__}')
    # Range checks
    for f in ['success_rate', 'confidence']:
        if f in mem and not (0.0 <= mem[f] <= 1.0):
            errors.append(f'{f} out of range [0,1]: {mem[f]}')
    if 'times_helpful' in mem and 'times_retrieved' in mem:
        if mem['times_helpful'] > mem['times_retrieved']:
            errors.append('times_helpful > times_retrieved')
    return errors

all_valid = True
for mem in memories:
    errs = validate_memory(mem)
    if errs:
        print(f"{mem['memory_id']}: {errs}")
        all_valid = False

if all_valid:
    print(f'All {len(memories)} memories pass schema validation.')

All 25 memories pass schema validation.


## 5. Summary

| Item | Detail |
|---|---|
| Task dataset | 80 problems, 4 categories, 20 each |
| Test runner | Validated on 5 canonical + 5 buggy samples |
| Synthetic memory bank | 25 memories: 20 high-SR, 5 low-SR |
| Schema validation | All 25 memories pass |

Outputs consumed by subsequent notebooks:
- `data/humaneval_subset.json`
- `data/synthetic_memories.json`